# Tugas 1 - Klasifikasi cacat produk

Eksperimen klasifikasi tujuh kelas menggunakan citra permukaan dan sinyal mesin. Notebook menghasilkan prediksi, bagian laporan, kurva, dan audit evaluasi.

## Sel 1 · Masukan dan keluaran

Dataset klasifikasi dibaca dari folder UTS. Ubah DATA bila letak Drive berbeda. HASIL adalah direktori kerja untuk prediksi dan bukti eksperimen.

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    if "DATA" not in globals() and not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    dasar_data = "/content/drive/MyDrive/UTS_DL/data_uts_dl"
    dasar_hasil = "/content/hasil_tugas1"
else:
    dasar_data = "data_uts_dl"
    dasar_hasil = "hasil_tugas1"

DATA = Path(globals().get("DATA", os.environ.get("UTS_DATA_DIR", dasar_data))).resolve()
HASIL = Path(globals().get("HASIL", os.environ.get("UTS_HASIL_TUGAS1", dasar_hasil))).resolve()
HASIL.mkdir(parents=True, exist_ok=True)
BUKTI = HASIL / "artifacts" / "tugas1"
BAGIAN = HASIL / "laporan_bagian" / "01_klasifikasi_cacat.pdf"

if not DATA.is_dir():
    raise FileNotFoundError(f"Periksa lokasi dataset: {DATA}")
print("Data:", DATA, "| Keluaran:", HASIL)

## Sel 2 · Fungsi bersama dan perangkat

Fungsi pembaca memeriksa kecocokan indeks metadata dengan array. Seed dan sidik SHA-256 dicatat untuk reproduksi. CPU empat thread adalah konfigurasi referensi. UTS_DEVICE dapat disetel ke cuda bila CUDA tersedia.

In [ ]:
%pip -q install reportlab pypdf

"""Pemuatan data, seed dan penyimpanan keluaran."""
import hashlib
import importlib.metadata
import json
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

SEED = (17, 29, 43)


def atur_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def baca_pasangan(folder, nama, id_kolom, modalitas):
    meta = pd.read_csv(folder / f"{nama}_meta.csv").sort_values("indeks").reset_index(drop=True)
    with np.load(folder / f"{nama}.npz", allow_pickle=False) as arsip:
        gambar, kedua = arsip["gambar"], arsip[modalitas]
    if not (len(meta) == len(gambar) == len(kedua)):
        raise ValueError(f"Panjang metadata dan array berbeda pada {nama}")
    if not meta[id_kolom].is_unique or not np.array_equal(meta.indeks, np.arange(len(meta))):
        raise ValueError(f"ID atau indeks tidak valid pada {nama}")
    return meta, gambar, kedua


def sidik(folder, nama):
    hasil = {}
    for file in nama:
        hash_file = hashlib.sha256()
        with (folder / file).open("rb") as stream:
            for bagian in iter(lambda: stream.read(1024 * 1024), b""):
                hash_file.update(bagian)
        hasil[file] = hash_file.hexdigest()
    return hasil


def versi():
    paket = ("torch", "numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "reportlab", "threadpoolctl")
    return {"Python": sys.version.split()[0], **{p: importlib.metadata.version(p) for p in paket}}


def simpan_json(path, isi):
    temp = path.with_suffix(".tmp")
    temp.write_text(json.dumps(isi, ensure_ascii=False, allow_nan=False, indent=2), encoding="utf-8")
    temp.replace(path)


def simpan_model(path, isi):
    temp = path.with_suffix(".tmp")
    torch.save(isi, temp)
    temp.replace(path)

DEVICE = torch.device(os.environ.get('UTS_DEVICE', 'cpu') if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cpu':
    torch.set_num_threads(4)
print('Perangkat:', DEVICE)

## Sel 3 · Eksplorasi, encoder, fusi, dan evaluasi

Pembagian latih, dev, dan evaluasi dibuat secara stratifikasi kelas dan lini. CNN 2D membaca citra, CNN 1D membaca sensor. Fusi fitur menggabungkan embedding. Fusi logit memakai mask kanal untuk mengatur bobot. Eksperimen membandingkan dua modalitas tunggal, label smoothing, augmentasi, dan sensor hilang. Tiga seed dipakai untuk model final. Audit perbedaan dengan label inspektur menghasilkan kandidat pemeriksaan.

In [ ]:
"""Klasifikasi cacat produk dari citra dan sinyal mesin."""
import json

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F


KELAS = ["normal", "gores", "noda", "retak", "gelembung", "las_dingin", "kontaminasi"]
EKSPERIMEN = {
    "citra": ("citra", 0.0, False),
    "sensor": ("sensor", 0.0, False),
    "fusi_fitur": ("fitur", 0.0, False),
    "fusi_logit": ("logit", 0.0, False),
    "logit_smoothing": ("logit", 0.10, False),
    "logit_augmentasi": ("logit", 0.10, True),
}
NAMA = {
    "citra": "CNN citra",
    "sensor": "CNN sensor",
    "fusi_fitur": "Fusi fitur",
    "fusi_logit": "Fusi logit",
    "logit_smoothing": "Fusi logit + smoothing",
    "logit_augmentasi": "Fusi logit + smoothing + augmentasi",
}
BERKAS = tuple(f"klasifikasi_{s}{e}" for s in ("train", "test") for e in (".npz", "_meta.csv"))


def persiapan(gambar, sensor, pusat, skala, device):
    peta = []
    for awal in range(0, len(gambar), 512):
        x = torch.as_tensor(gambar[awal:awal+512], device=device).float().unsqueeze(1)
        x = (x - x.mean((2, 3), keepdim=True)) / x.std((2, 3), keepdim=True).clamp_min(5)
        detail = x - F.avg_pool2d(x, 5, stride=1, padding=2, count_include_pad=False)
        peta.append(torch.cat([F.avg_pool2d(x, 2), F.avg_pool2d(detail, 2),
                               F.max_pool2d(detail, 2), F.max_pool2d(-detail, 2)], dim=1).half())
    x = (sensor.astype(np.float32) - pusat) / skala
    mask = np.isfinite(x).any(2)
    x = np.nan_to_num(x, nan=0).clip(-12, 12)
    return (torch.cat(peta).contiguous(memory_format=torch.channels_last),
            torch.as_tensor(x, device=device), torch.as_tensor(mask, device=device))


def blok_citra(masuk, keluar, stride):
    return nn.Sequential(nn.Conv2d(masuk, keluar, 3, stride, 1, bias=False),
                         nn.BatchNorm2d(keluar), nn.SiLU())


class Cacat(nn.Module):
    def __init__(self, mode):
        super().__init__()
        self.mode = mode
        if mode != "sensor":
            self.citra = nn.Sequential(blok_citra(4, 24, 2), blok_citra(24, 32, 2),
                                       blok_citra(32, 48, 1), blok_citra(48, 64, 2))
            self.ringkas_citra = nn.Sequential(nn.Linear(128, 80), nn.LayerNorm(80), nn.SiLU())
        if mode != "citra":
            self.sensor = nn.Sequential(
                nn.Conv1d(6, 24, 9, 2, 4), nn.BatchNorm1d(24), nn.SiLU(),
                nn.Conv1d(24, 32, 5, 2, 2), nn.BatchNorm1d(32), nn.SiLU(),
                nn.Conv1d(32, 40, 5, 2, 2), nn.BatchNorm1d(40), nn.SiLU())
            self.ringkas_sensor = nn.Sequential(nn.Linear(187, 80), nn.LayerNorm(80), nn.SiLU())
        if mode in ("citra", "logit"):
            self.kepala_citra = nn.Linear(80, len(KELAS))
        if mode in ("sensor", "logit"):
            self.kepala_sensor = nn.Linear(80, len(KELAS))
        if mode == "fitur":
            self.kepala_fusi = nn.Sequential(nn.Linear(163, 96), nn.LayerNorm(96), nn.SiLU(),
                                            nn.Dropout(0.15), nn.Linear(96, len(KELAS)))
        if mode == "logit":
            self.gate = nn.Linear(3, len(KELAS))

    def forward(self, gambar, sensor, mask):
        if self.mode != "sensor":
            g = self.citra(gambar)
            g = self.ringkas_citra(torch.cat([g.mean((2, 3)), g.amax((2, 3))], 1))
        if self.mode == "citra":
            return self.kepala_citra(g)
        diff = torch.diff(sensor, dim=2, prepend=sensor[:, :, :1])
        s = self.sensor(torch.cat([sensor, diff], 1))
        mean = sensor.reshape(-1, 3, 4, 32).mean(-1).flatten(1)
        rms = diff.reshape(-1, 3, 4, 32).square().mean(-1).sqrt().flatten(1)
        valid = mask.to(sensor.dtype)
        s = self.ringkas_sensor(torch.cat([F.adaptive_avg_pool1d(s, 4).flatten(1), mean, rms, valid], 1))
        if self.mode == "sensor":
            return self.kepala_sensor(s)
        tersedia = mask.any(1, keepdim=True)
        if self.mode == "fitur":
            return self.kepala_fusi(torch.cat([g, s * tersedia, valid], 1))
        w = self.gate(valid).sigmoid() * tersedia
        return (1 - w) * self.kepala_citra(g) + w * self.kepala_sensor(s)


def macro(y, p):
    return float(f1_score(y, p, labels=np.arange(len(KELAS)), average="macro", zero_division=0))


@torch.inference_mode()
def prediksi(model, data, indeks, hilang=False):
    model.eval()
    hasil = []
    for i in indeks.split(256):
        g, s, m = (x[i] for x in data)
        if hilang:
            s, m = torch.zeros_like(s), torch.zeros_like(m)
        hasil.append(model(g.float(), s, m).softmax(1).cpu().numpy())
    return np.concatenate(hasil)


def latih(nama, seed, data, y, fit, dev, device, epoch, out, identitas, sanity=False):
    mode, smoothing, augmentasi = EKSPERIMEN[nama]
    atur_seed(seed)
    model = Cacat(mode).to(device, memory_format=torch.channels_last)
    path = out / f"{nama}_{seed}.pt"
    konfigurasi = [identitas, epoch, seed, nama, str(device), 1]
    if not sanity and path.is_file():
        ckpt = torch.load(path, map_location="cpu", weights_only=True)
        if ckpt["konfigurasi"] == konfigurasi:
            model.load_state_dict(ckpt["model"])
            return model, ckpt["kurva"]
    jumlah = torch.bincount(y[fit], minlength=len(KELAS)).float().clamp_min(1)
    bobot = (jumlah.mean() / jumlah).pow(0.5)
    opt = torch.optim.AdamW(model.parameters(), lr=0.0025, weight_decay=0.001)
    jadwal = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epoch, eta_min=0.00035)
    y_dev = y[dev].cpu().numpy()
    satu_batch = torch.cat([fit[y[fit] == k][:8] for k in range(len(KELAS))])
    terbaik, state, kurva = -1.0, None, []
    for e in range(epoch):
        model.train()
        total = 0.0
        batch = [satu_batch] if sanity else fit[torch.randperm(len(fit), device=device)].split(256)
        for i in batch:
            g, s, m = (x[i] for x in data)
            g = g.float()
            if mode in ("fitur", "logit") and not sanity:
                m = m & (torch.rand((len(i), 1), device=device) >= 0.10)
                s = s * m.unsqueeze(2)
            if augmentasi:
                flip = torch.rand((len(i), 1, 1, 1), device=device) < 0.5
                g = torch.where(flip, g.flip((-1 if e % 2 else -2,)), g)
                g = g * (0.85 + 0.30 * torch.rand((len(i), 1, 1, 1), device=device))
                s = s + 0.12 * torch.randn((len(i), 3, 1), device=device) * m.unsqueeze(2)
            p = model(g, s, m)
            loss = F.cross_entropy(p, y[i], weight=None if sanity else bobot,
                                   label_smoothing=0.0 if sanity else smoothing)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            total += loss.item() * len(i)
        if sanity:
            kurva.append({"langkah": e+1, "loss": total / len(satu_batch),
                          "akurasi": float((p.argmax(1) == y[satu_batch]).float().mean())})
            continue
        jadwal.step()
        f1 = macro(y_dev, prediksi(model, data, dev).argmax(1))
        kurva.append({"epoch": e+1, "loss": total / len(fit), "f1_dev": f1})
        if f1 > terbaik:
            terbaik = f1
            state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"T1 | {nama} | seed {seed} | epoch {e+1}/{epoch} | F1 dev {f1:.4f}", flush=True)
    if not sanity:
        simpan_model(path, {"konfigurasi": konfigurasi, "model": state, "kurva": kurva})
        model.load_state_dict(state)
    return model, kurva


def evaluasi(y, p, lini, kanal):
    def subset(mask):
        return {"n": int(mask.sum()), "f1": macro(y[mask], p[mask])}
    return {
        "f1": macro(y, p),
        "per_kelas": dict(zip(KELAS, f1_score(y, p, labels=np.arange(len(KELAS)),
                                            average=None, zero_division=0).tolist())),
        "per_lini": {l: subset(lini == l) for l in sorted(set(lini))},
        "per_sensor": {str(k): subset(kanal == k) for k in sorted(set(kanal))},
    }


def eksplorasi(meta, gambar, sensor, out):
    fig, ax = plt.subplots(7, 5, figsize=(10, 8), layout="constrained")
    lengkap = np.isfinite(sensor).all((1, 2))
    for j, kelas in enumerate(KELAS):
        contoh = np.flatnonzero((meta.kelas == kelas) & lengkap)[:2]
        for i, n in enumerate(contoh):
            ax[j, i].imshow(gambar[n], cmap="gray", vmin=60, vmax=200)
            ax[j, i].axis("off")
            for kanal in range(3):
                ax[j, kanal+2].plot(sensor[n, kanal], lw=0.65)
        ax[j, 0].set_title(kelas, fontsize=9)
        for a in ax[j, 2:]:
            a.tick_params(labelsize=7)
    for a, k in zip(ax[0, 2:], ["Suhu", "Getaran", "Arus"]):
        a.set_title(k)
    fig.savefig(out / "contoh.png", dpi=135)
    plt.close(fig)


def visual_hasil(out, final, y, p):
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.1), layout="constrained")
    for seed in SEED:
        riwayat = pd.read_csv(out / f"kurva_{final}_{seed}.csv")
        ax[0].plot(riwayat.epoch, riwayat.f1_dev, marker=".", label=f"F1 dev {seed}")
    loss = ax[0].twinx()
    loss.plot(riwayat.epoch, riwayat.loss, "--", color="0.5")
    loss.set_ylabel(f"Loss latih seed {SEED[-1]}")
    ax[0].set(xlabel="Epoch", ylabel="Macro-F1 dev")
    ax[0].legend(loc="lower right", fontsize=8)
    cm = confusion_matrix(y, p, labels=np.arange(len(KELAS)), normalize="true")
    ax[1].imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax[1].set(xticks=range(7), xticklabels=KELAS, yticks=range(7), yticklabels=KELAS,
              xlabel="Prediksi", ylabel="Label inspektur")
    ax[1].tick_params(axis="x", labelrotation=38, labelsize=8)
    ax[1].tick_params(axis="y", labelsize=8)
    for i, j in np.ndindex(cm.shape):
        ax[1].text(j, i, f"{cm[i,j]:.2f}", ha="center", va="center", fontsize=7,
                   color="white" if cm[i,j] > 0.58 else "black")
    fig.savefig(out / "hasil.png", dpi=145)
    plt.close(fig)


def jalankan(data_dir, out, hasil, device, cepat=False):
    model_dir = out / "model"
    model_dir.mkdir(parents=True, exist_ok=True)
    hash_data = sidik(data_dir, BERKAS)
    m, g, s = baca_pasangan(data_dir, "klasifikasi_train", "id_unit", "sensor")
    mt, gt, st = baca_pasangan(data_dir, "klasifikasi_test", "id_unit", "sensor")
    if set(m.id_unit) & set(mt.id_unit):
        raise ValueError("ID klasifikasi latih dan uji bertumpang tindih")
    eksplorasi(m, g, s, out)
    y = m.kelas.map({k: i for i, k in enumerate(KELAS)}).to_numpy(dtype=np.int64)
    strata = m.kelas + "_" + m.lini
    fit, sisa = train_test_split(np.arange(len(m)), test_size=.2, stratify=strata, random_state=2026)
    a, b = train_test_split(sisa, test_size=.5, stratify=strata.iloc[sisa], random_state=2027)
    if cepat:
        fit, _ = train_test_split(fit, train_size=1500, stratify=strata.iloc[fit], random_state=17)
        a, b = a[:450], b[:450]
    split = m[["id_unit"]].assign(split="tidak dipakai")
    for k, i in (("latih", fit), ("dev", a), ("evaluasi", b)):
        split.loc[i, "split"] = k
    split.to_csv(out / "pembagian.csv", index=False)
    subset = s[fit].astype(np.float32)
    pusat = np.nanmean(subset, axis=0)
    skala = np.nanstd(subset, axis=0).clip(.05)
    del subset
    np.savez(out / "normalisasi.npz", pusat=pusat, skala=skala)
    data, test = persiapan(g, s, pusat, skala, device), persiapan(gt, st, pusat, skala, device)
    del g, gt
    indeks = lambda i: torch.as_tensor(i, device=device)
    yt, ft, dev, ev = map(indeks, (y, fit, a, b))
    epoch = 1 if cepat else 9
    identitas = json.dumps({"data": hash_data, "cepat": cepat}, sort_keys=True)
    _, sanity = latih("fusi_fitur", SEED[0], data, yt, ft, dev, device, 65, model_dir, identitas, True)
    pd.DataFrame(sanity).to_csv(out / "sanity.csv", index=False)
    kanal = np.isfinite(s).any(2).sum(1)
    kanal_test = np.isfinite(st).any(2).sum(1)
    lini = m.lini.to_numpy()
    ab, model_awal = {}, {}
    for nama in EKSPERIMEN:
        model, kurva = latih(nama, SEED[0], data, yt, ft, dev, device, epoch, model_dir, identitas)
        model_awal[nama] = model
        pd.DataFrame(kurva).to_csv(out / f"kurva_{nama}_{SEED[0]}.csv", index=False)
        ab[nama] = evaluasi(y[b], prediksi(model, data, ev).argmax(1), lini[b], kanal[b])
        ab[nama]["dev"] = max(x["f1_dev"] for x in kurva)
    final = max(list(EKSPERIMEN)[2:], key=lambda k: ab[k]["dev"])
    lengkap = b[kanal[b] == 3]
    prob, prob_uji, prob_drop, antar_seed = [], [], [], []
    it = indeks(np.arange(len(mt)))
    for seed in SEED:
        if seed == SEED[0]:
            model = model_awal[final]
        else:
            model, kurva = latih(final, seed, data, yt, ft, dev, device, epoch, model_dir, identitas)
            pd.DataFrame(kurva).to_csv(out / f"kurva_{final}_{seed}.csv", index=False)
        prob.append(prediksi(model, data, ev))
        prob_drop.append(prediksi(model, data, indeks(lengkap), hilang=True))
        prob_uji.append(prediksi(model, test, it))
        antar_seed.append({"seed": seed, "f1": macro(y[b], prob[-1].argmax(1))})
    nama_kelas = np.array(KELAS)
    peluang = np.mean(prob, axis=0)
    p = peluang.argmax(1)
    pd.DataFrame({"id_unit": mt.id_unit,
                  "kelas": nama_kelas[np.mean(prob_uji, axis=0).argmax(1)]
                  }).to_csv(hasil / "prediksi_klasifikasi.csv", index=False)
    audit = pd.DataFrame({"id_unit": m.id_unit.iloc[b], "inspektur": nama_kelas[y[b]],
                          "prediksi": nama_kelas[p], "peluang": peluang.max(1), "lini": lini[b],
                          "kanal_sensor": kanal[b]})
    audit.to_csv(out / "evaluasi.csv", index=False)
    konflik = audit[audit.inspektur != audit.prediksi].sort_values("peluang", ascending=False).copy()
    konflik["di_atas_ambang"] = konflik.peluang >= 0.85
    konflik.head(max(20, int(konflik.di_atas_ambang.sum()))).to_csv(out / "audit_label.csv", index=False)
    visual_hasil(out, final, y[b], p)
    def ringkas(meta, k):
        return {l: {"n": int((meta.lini == l).sum()),
                    "hilang": int(((meta.lini == l) & (k == 0)).sum()),
                    "parsial": int(((meta.lini == l) & (k > 0) & (k < 3)).sum()),
                    "kecerahan_rerata": float(meta.loc[meta.lini == l, "intensitas"].mean())}
                for l in sorted(meta.lini.unique())}
    for meta, file in ((m, "klasifikasi_train"), (mt, "klasifikasi_test")):
        with np.load(data_dir / f"{file}.npz") as archive:
            meta["intensitas"] = archive["gambar"].mean((1, 2))
    simpan_json(out / "hasil.json", {
        "cepat": cepat, "epoch": epoch, "device": str(device), "versi": versi(), "hash": hash_data,
        "split": {"latih": len(fit), "dev": len(a), "evaluasi": len(b)},
        "latih": ringkas(m, kanal), "uji": ringkas(mt, kanal_test),
        "kelas_lini": pd.crosstab(m.kelas, m.lini).to_dict(),
        "ablasi": ab, "final": final, "seed": antar_seed, "sanity": [sanity[0], sanity[-1]],
        "ensemble": evaluasi(y[b], p, lini[b], kanal[b]),
        "audit": {"ambang": 0.85, "jumlah": int(konflik.di_atas_ambang.sum())},
        "stress": {"n": len(lengkap), "asli": macro(y[lengkap], p[kanal[b] == 3]),
                   "dihilangkan": macro(y[lengkap], np.mean(prob_drop, axis=0).argmax(1))},
    })

## Sel 4 · Bagian laporan dan verifikasi

Bagian PDF berisi eksplorasi, hasil ablasi, kurva pelatihan, serta confusion matrix. Pemeriksaan menghitung ulang macro-F1 dari data evaluasi, mencocokkan ID uji, dan memeriksa konsistensi dataset.

In [ ]:
"""Laporan Bahasa Indonesia, dirakit dari hasil eksperimen terverifikasi."""
from xml.sax.saxutils import escape

from reportlab.lib import colors
from reportlab.lib.enums import TA_LEFT
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle
from reportlab.platypus import Image, PageBreak, Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle



BIRU = colors.HexColor("#173452")
TEAL = colors.HexColor("#166254")
TEKS = colors.HexColor("#202b36")
PUCAT = colors.HexColor("#edf4f7")
ABU = colors.HexColor("#546476")
W = 505
STYLE = {
    "kicker": ParagraphStyle("kicker", fontName="Helvetica-Bold", fontSize=9, leading=12,
                             textColor=TEAL, spaceAfter=8),
    "h1": ParagraphStyle("h1", fontName="Helvetica-Bold", fontSize=22, leading=25,
                         textColor=BIRU, spaceAfter=11),
    "h2": ParagraphStyle("h2", fontName="Helvetica-Bold", fontSize=12, leading=15,
                         textColor=BIRU, spaceBefore=9, spaceAfter=5),
    "p": ParagraphStyle("p", fontName="Helvetica", fontSize=9.4, leading=13.7,
                        textColor=TEKS, alignment=TA_LEFT, spaceAfter=7),
    "caption": ParagraphStyle("caption", fontName="Helvetica", fontSize=8.2, leading=11.5,
                              textColor=ABU, spaceBefore=4, spaceAfter=8),
    "cell": ParagraphStyle("cell", fontName="Helvetica", fontSize=8.4, leading=10.7, textColor=TEKS),
    "label": ParagraphStyle("label", fontName="Helvetica-Bold", fontSize=8.4, leading=10.7,
                            textColor=colors.white),
}


def angka(x, digit=3):
    return f"{x:,.{digit}f}".replace(",", "~").replace(".", ",").replace("~", ".")


def n(x):
    return angka(x, 0)


def pct(x, digit=1):
    return angka(x * 100, digit) + "%"


def footer(canvas, doc):
    canvas.setTitle("UTS Deep Learning | Klasifikasi, RUL, dan Klaster")
    canvas.setAuthor("Kelompok UTS Deep Learning")
    canvas.setStrokeColor(colors.HexColor("#d8e3e9"))
    canvas.line(45, 40, 550, 40)
    canvas.setFillColor(ABU)
    canvas.setFont("Helvetica", 8)
    canvas.drawString(45, 27, "UTS DEEP LEARNING  /  EKSPERIMEN MULTIMODAL")
    canvas.drawRightString(550, 27, f"T1 / {doc.page}")


from pypdf import PdfReader


def buat_bagian(out, k, target):
    art = out / 'artifacts'
    story = []
    def p(s, gaya="p"):
        story.append(Paragraph(s, STYLE[gaya]))

    def judul(bagian, h):
        p(bagian, "kicker")
        p(h, "h1")

    def sub(s):
        p(s, "h2")

    def tabel(header, rows, lebar):
        isi = [[Paragraph(escape(str(s)), STYLE["label"]) for s in header]]
        isi += [[Paragraph(escape(str(s)), STYLE["cell"]) for s in row] for row in rows]
        t = Table(isi, colWidths=lebar, repeatRows=1, hAlign="LEFT")
        t.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, 0), BIRU),
                               ("ROWBACKGROUNDS", (0, 1), (-1, -1), [PUCAT, colors.white]),
                               ("VALIGN", (0, 0), (-1, -1), "TOP"),
                               ("LEFTPADDING", (0, 0), (-1, -1), 7),
                               ("RIGHTPADDING", (0, 0), (-1, -1), 7),
                               ("TOPPADDING", (0, 0), (-1, -1), 5),
                               ("BOTTOMPADDING", (0, 0), (-1, -1), 5)]))
        story.append(t)
        story.append(Spacer(1, 7))

    def figur(tugas, nama, hmax, ket):
        im = Image(str(art / f"tugas{tugas}" / f"{nama}.png"))
        skala = min(W / im.imageWidth, hmax / im.imageHeight)
        im.drawWidth, im.drawHeight = im.imageWidth*skala, im.imageHeight*skala
        im.hAlign = "CENTER"
        story.append(im)
        p(ket, "caption")

    def halaman():
        story.append(PageBreak())

    final_k = NAMA[k["final"]]
    judul("TUGAS 1 / DATA", "Citra permukaan dan rekaman sensor")
    rows = [[set_, l, n(v["n"]), n(v["hilang"]), n(v["parsial"]), angka(v["kecerahan_rerata"], 1)]
            for set_ in ("latih", "uji") for l, v in k[set_].items()]
    tabel(["Set", "Lini", "Unit", "Sensor kosong", "Sensor parsial", "Rerata intensitas"],
          rows, [61, 44, 77, 103, 103, 117])
    a = k["latih"]["B"]["n"] / sum(v["n"] for v in k["latih"].values())
    b = k["uji"]["B"]["n"] / sum(v["n"] for v in k["uji"].values())
    p(f"Proporsi lini B bergeser dari {pct(a)} pada latih menjadi {pct(b)} pada uji. "
      "Gores dan retak dapat muncul sebagai struktur tipis di citra. Rekaman tekanan memperlihatkan pola "
      "arus naik, plateau, lalu turun. Suhu dan getaran bervariasi antarunit, sehingga satu contoh belum "
      "cukup untuk mengaitkan pola tertentu dengan cacat.")
    tabel(["Lini"] + KELAS, [[l] + [n(v[nama]) for nama in KELAS] for l, v in k["kelas_lini"].items()],
          [42, 63, 57, 54, 54, 76, 72, 87])
    figur(1, "contoh", 343, "Dua unit per kelas dengan sensor lengkap. Kolom sinyal menumpuk dua "
         "rekaman pada sumbu waktu sampel. Batas vertikal menyesuaikan kanal dan baris.")
    p("Normalisasi citra menggunakan rata-rata dan simpangan baku tiap unit. Citra 64 x 64 diringkas "
      "menjadi empat peta 32 x 32 yang membawa intensitas, residu lokal, dan dua ekstrem residu. "
      "Sinyal distandardisasi per kanal dan posisi waktu dari subset latih. NaN diisi nol pada skala "
      "standar, dengan mask tiga kanal.")

    halaman()
    judul("TUGAS 1 / MODEL", "Fusi dan ketahanan terhadap gangguan")
    s = k["split"]
    p(f"Pembagian stratifikasi kelas dan lini berisi {n(s['latih'])} latih, {n(s['dev'])} pengembangan, "
      f"dan {n(s['evaluasi'])} evaluasi. CNN 2D membaca citra, CNN 1D membaca sinyal dan beda waktu. "
      "Fusi fitur menggabungkan embedding sebelum kepala klasifikasi. Fusi logit mencampur keluaran "
      f"dua kepala memakai bobot per kelas dari mask. Pelatihan berjalan {k['epoch']} epoch per konfigurasi. "
      "Semua fusi menyembunyikan sensor pada 10% sampel latih secara acak.")
    tabel(["Eksperimen / seed 17", "F1 dev", "F1 evaluasi", "Lini A", "Lini B"], [
        [NAMA[nama], angka(v["dev"]), angka(v["f1"]), angka(v["per_lini"]["A"]["f1"]),
         angka(v["per_lini"]["B"]["f1"])] for nama, v in k["ablasi"].items()
    ], [233, 68, 70, 67, 67])
    aa = k["ablasi"]
    delta_s = aa["logit_smoothing"]["f1"] - aa["fusi_logit"]["f1"]
    delta_a = [aa["logit_augmentasi"]["per_lini"][l]["f1"] -
               aa["logit_smoothing"]["per_lini"][l]["f1"] for l in ("A", "B")]
    p(f"Smoothing 0,10 mengubah F1 evaluasi sebesar {angka(delta_s)}. Augmentasi balik citra, perubahan "
      f"kontras dan drift sensor mengubah F1 lini A sebesar {angka(delta_a[0])} dan lini B sebesar "
      f"{angka(delta_a[1])} terhadap smoothing saja. Ada {k['audit']['jumlah']} konflik label dengan "
      "probabilitas model minimal 0,85 pada evaluasi. Kandidat utama masuk audit_label.csv untuk pemeriksaan visual.")
    tabel(["F1 evaluasi"] + KELAS, [
        [label] + [angka(val[nama], 2) for nama in KELAS]
        for label, val in [("Citra", aa["citra"]["per_kelas"]), ("Sensor", aa["sensor"]["per_kelas"]),
                           ("Ensemble", k["ensemble"]["per_kelas"])]
    ], [72, 55, 53, 49, 48, 71, 73, 84])
    cit = aa["citra"]["per_kelas"]
    sen = aa["sensor"]["per_kelas"]
    untung_c = max(KELAS, key=lambda l: cit[l]-sen[l])
    untung_s = max(KELAS, key=lambda l: sen[l]-cit[l])
    p(f"Selisih citra minus sensor terbesar muncul pada <b>{untung_c}</b> ({angka(cit[untung_c]-sen[untung_c])}). "
      f"Selisih sensor minus citra terbesar muncul pada <b>{untung_s}</b> ({angka(sen[untung_s]-cit[untung_s])}).")
    ket_seed = ", ".join(f"{v['seed']} ({angka(v['f1'])})" for v in k["seed"])
    st = k["stress"]
    kondisi = " | ".join(f"{j} kanal: {angka(v['f1'])} (n={n(v['n'])})"
                        for j, v in k["ensemble"]["per_sensor"].items())
    lini = " dan ".join(f"{l} {angka(v['f1'])}" for l, v in k["ensemble"]["per_lini"].items())
    p(f"<b>{final_k}</b> terpilih dari F1 dev. Ensemble memakai rata-rata probabilitas dan memperoleh F1 lini "
      f"{lini}. F1 per seed: {ket_seed}. Evaluasi menurut sensor: {kondisi}. Menghapus seluruh sensor secara "
      f"simulasi pada {n(st['n'])} unit lengkap mengubah F1 dari {angka(st['asli'])} ke "
      f"{angka(st['dihilangkan'])}. Pada sanity check fusi fitur, akurasi satu batch berubah dari "
      f"{pct(k['sanity'][0]['akurasi'])} ke {pct(k['sanity'][1]['akurasi'])}.")
    figur(1, "hasil", 150, "Kurva pengembangan tiga seed, loss satu seed, dan confusion matrix ensemble.")
    p("Label inspektur bisa mengandung salah label. Perbedaan skor antar lini juga dapat dipengaruhi "
      "komposisi dan tingkat kesulitan unit. Split mengukur kedua lini yang sudah terlihat selama pelatihan.", "caption")
    doc = SimpleDocTemplate(str(target), pagesize=A4, leftMargin=45, rightMargin=45,
                            topMargin=40, bottomMargin=56)
    doc.build(story, onFirstPage=footer, onLaterPages=footer)

def verifikasi_tugas1(ringkas, pdf):
    pred = pd.read_csv(HASIL / "prediksi_klasifikasi.csv")
    meta = pd.read_csv(DATA / "klasifikasi_test_meta.csv")
    assert list(pred.columns) == ["id_unit", "kelas"]
    assert pred.id_unit.is_unique and not pred.isna().any().any()
    assert set(pred.id_unit) == set(meta.id_unit)
    assert set(pred.kelas) <= set(KELAS)
    ev = pd.read_csv(BUKTI / "evaluasi.csv")
    split = pd.read_csv(BUKTI / "pembagian.csv")
    assert split.id_unit.is_unique
    assert set(ev.id_unit) == set(split.loc[split.split == "evaluasi", "id_unit"])
    f1 = f1_score(ev.inspektur, ev.prediksi, labels=KELAS, average="macro", zero_division=0)
    np.testing.assert_allclose(f1, ringkas["ensemble"]["f1"], rtol=0, atol=1e-12)
    assert sidik(DATA, BERKAS) == ringkas["hash"]
    assert len(PdfReader(pdf).pages) == 2
    print(f"Tugas 1 LULUS | {len(pred):,} unit uji | macro-F1 evaluasi {f1:.6f}")
    return {"status": "LULUS", "baris": len(pred), "macro_f1": float(f1)}

## Sel 5 · Eksekusi Tugas 1

Jalankan seluruh alur dengan jalankan_tugas1(). Dalam checkout repositori kelompok, checkpoint CPU dengan konfigurasi cocok dapat dipakai kembali. Jika checkpoint belum tersedia, model dilatih dari data UTS.

In [ ]:
from shutil import copy2
from threadpoolctl import threadpool_limits


def muat_checkpoint_referensi():
    lokasi = globals().get("REPO")
    if lokasi is None:
        kandidat = [Path.cwd(), *Path.cwd().parents]
        lokasi = next((p for p in kandidat if (p / "kode" / "tugas1" / "model").is_dir()), None)
    if lokasi is None or DEVICE.type != "cpu":
        return
    sumber = Path(lokasi) / "kode" / "tugas1" / "model"
    target = BUKTI / "model"
    target.mkdir(parents=True, exist_ok=True)
    for berkas in sumber.glob("*.pt"):
        keluaran = target / berkas.name
        if not keluaran.exists():
            copy2(berkas, keluaran)


def jalankan_tugas1():
    muat_checkpoint_referensi()
    torch.set_num_threads(4)
    with threadpool_limits(limits=4):
        jalankan(DATA, BUKTI, HASIL, DEVICE)
    ringkas = json.loads((BUKTI / "hasil.json").read_text(encoding="utf-8"))
    BAGIAN.parent.mkdir(parents=True, exist_ok=True)
    buat_bagian(HASIL, ringkas, BAGIAN)
    hasil = verifikasi_tugas1(ringkas, BAGIAN)
    simpan_json(BUKTI / "verifikasi.json", hasil)
    return hasil


hasil_tugas1 = jalankan_tugas1()

## Sel 6 · Ekspor berkas

Arsip memuat prediksi_klasifikasi.csv, PDF bagian, dan tabel atau grafik eksperimen. CSV ditempatkan di root pengumpulan.

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

arsip = HASIL.parent / "hasil_tugas1.zip"
with ZipFile(arsip, "w", compression=ZIP_DEFLATED) as paket:
    for rel in ["prediksi_klasifikasi.csv", "laporan_bagian/01_klasifikasi_cacat.pdf"]:
        paket.write(HASIL / rel, rel)
    for sumber in sorted(BUKTI.glob("*")):
        if sumber.is_file():
            paket.write(sumber, f"bukti_eksperimen/tugas1/{sumber.name}")
print("Arsip:", arsip)
if drive is not None:
    from google.colab import files
    files.download(str(arsip))